# Synthesis and property extraction with Qwen

Use the **extract** kernel. Run cells in order: list a paper's samples together with the properties the property-presence classifier found for that paper, choose a sample ID, then send the paper and sample to Qwen. Synthesis and property extraction are sent from separate cells, so either can be run or changed on its own. Prompts are editable below. Nothing is saved; the database and papers are only read.

The synthesis and property prompts have separate names (`SYNTH_*` and `TG_*`), so neither overwrites the other.


## The 20 papers

Sample counts include **all sample records in the database** for each paper, not just the one sample selected for the pilot. Counts read on 20 October 2026.

| Paper UID | No. of samples | Title |
|---|---:|---|
| ID001100849 | 5 | Physics Augmented Machine Learning Discovery of Composition-Dependent Constitutive Laws for 3D Printed Digital Materials |
| ID001098910 | 7 | Polycaprolactone/graphite nanoplates composite nanopapers |
| ID001098286 | 6 | Interfacial Entropic Interactions Tunes Fragility and Dynamic Heterogeneity of Glassy Athermal Polymer Nanocomposite films |
| ID001098538 | 4 | Magnetic Shape Memory Polymers with Integrated Multifunctional Shape Manipulations |
| ID000573272 | 10 | Continuous glycolysis of waste PET via reactive extrusion and copolymerization with PTMG for enhanced toughness |
| ID000415115 | 6 | Effect of chemically modified lignin addition on the physicochemical properties of PCL nanofibers |
| ID000588793 | 11 | Structure-property relationships and microstructural origins of elasticity in Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymers and their fibers |
| ID000581823 | 8 | Thermal properties, rheology and foams of polystyrene-block-poly(4-vinylpyridine) diblock copolymers |
| ID001002844 | 20 | Dispersant and Protective Roles of Amphiphilic Poly(ethylene phosphate) Block Copolymers in Polyester/Bone Mineral Composites. |
| ID001032337 | 6 | Biobased Thermoplastic Elastomers: Structure-Property Relationship of Poly(hexamethylene 2,5-furanodicarboxylate)-Block-Poly(tetrahydrofuran) Copolymers Prepared by Melt Polycondensation. |
| ID000752848 | 5 | Trans crystallization behavior and strong reinforcement effect of cellulose nanocrystals on reinforced poly(butylene succinate) nanocomposites. |
| ID000180835 | 2 | A comparative investigation of uranium and thorium adsorption behavior on amidoximated copolymeric hydrogel |
| ID000180254 | 6 | Flexible Conductive Nanocomposites for Electrochemical Devices Based on Chlorinated Natural Rubber/Nickel Oxide Nanoparticles |
| ID000750264 | 2 | A surface-grafted thermoresponsive hydrogel in which the surface structure dominates the bulk properties. |
| ID000732480 | 6 | Enhancing the degradability of poly(lactic acid) through the introduction of main-chain thionoester linkages by ring-opening copolymerization |
| ID000732356 | 16 | Ring-opening polymerization of six-membered cyclic hybrid dimers composed of an oxoester and thioester |
| ID000905310 | 24 | Green revolution in polymer composites: performance and degradability of natural fiber reinforced biopolymers |
| ID000137005 | 7 | A study of oriented conductive composites with segregated network structure obtained via solid-state processing of UHMWPE reactor powder and carbon nanofillers |
| ID000143254 | 4 | Synthesis of waterborne acrylic copolymer resin as a binding agent for the development of water-based inks in the printing application |
| ID000061162 | 4 | Antibacterial and thermomechanical properties of composites of polylactic acid modified with capsicum oleoresin-impregnated nanoporous silica |

## Why the prompt excludes results, labels and test steps

The synthesis paragraph is **predictor input** for property prediction. The model should see only what is known *before* the material is measured: starting materials, amounts, conditions and processing. If the text contains anything that is only known after making and measuring the material, the model is handed part of the answer.

### What leaked in earlier runs (v5–v7, 8 + 7 + 20 samples)

| Type | Example from the runs | Status |
|---|---|---|
| Measured outcome | "91% conversion, 98% selectivity", "mean degree of polymerization of 20", "quantitative deprotection" (P1d polycarbonate) | v5 no-think; v6 no-think still kept 91% and "quantitative" |
| Measured value of an intermediate | "oligomers (Mn = 1265 Da)" (recycled PET); "grafting density 1.3 chains/nm²"; "80 wt% PS / 20 wt% P4VP" (measured composition); "crude precursor (8.77 g, 36.4 mmol)"; "until complete consumption of the monomers" | still present in v6/v7 |
| Appearance | "dense light yellowish liquid", "highly viscous prepolymer", "clear viscous solution", "homogeneous slurry" | v5; mostly removed in v6/v7 |
| Sample label | "End5700", heading "plasticized methacrylate compleximer", "nwLLZO", "regenerated PET", "SiCO" | occasional in v6/v7 |
| Test-only steps | 48 h conditioning at 23 °C/50% RH, Teflon pre-crack film for a fracture test, transfer onto test wafers, tensile-bar mould names | occasional, mostly no-think |

### Why each type harms property prediction

1. **Direct target leakage.** Mn, Mw, dispersity and DP are target properties. "DP 20" or "Mn 1265 Da" in the input gives the answer (or a strong proxy) away.
2. **Indirect leakage through correlated outcomes.** Conversion, yield and "complete consumption" track molecular weight; grafting density and measured composition track Tg, modulus and dispersion; appearance words carry hidden property information ("viscous" → molecular weight, "transparent" → crystallinity, "homogeneous" → better mechanical behaviour, "brittle" → elongation).
3. **Labels encode answers and paper identity.** Many codes contain results: P4-96K (Mn 96 kg/mol), PS80-b-P4VP20¹⁶⁴ (composition and Mn), c-PMMA225 (DP). Series codes (PL05, DM-60) reveal the position in a series. Unique labels let the model recognise the paper and memorise its value range instead of learning chemistry.
4. **Test-only steps add noise or reveal the target.** Conditioning, notching or a crack starter describe how a property was measured, not how the material was made. Identical materials would get different texts, and a test step can reveal which property label goes with the sample.
5. **Gap comments are spurious features.** Phrases like "temperature not reported" depend on writing style, not chemistry, and give the model meaningless patterns to learn.

**Consequence:** the use case is predicting a property *before* synthesis, when there is no yield, no measured Mn, no observed colour and no sample code. A model trained on leaky texts looks accurate on held-out papers (which leak in the same way) and fails on genuinely new recipes. Removing these items is what makes the reported test performance meaningful.

### What must stay

Actual input amounts carried into the next step (the 1.0 g of polymer redissolved, not the 85% yield), formulation compositions and loadings (80 wt% PLA / 20 wt% PEG, not "P20"), stated target or nominal recipe parameters, starting-material specifications, and process endpoints that are stopping criteria of the procedure ("until 95% of the theoretical water was distilled off", "until the agitator power reached 40 W").

### Open cases to settle in the prompt

- **Intermediate identified only by a measured value** (recycled PET: the oligomer is only called "Mn = 1265 Da"): describe it by how it was made and drop the measured value.
- **Endpoint vs outcome:** "until the water was distilled off" stays; "until complete consumption" / "to achieve 91% conversion" goes.
- **Values the authors estimated from measurements** (e.g. volume or mass fractions calculated from SAXS sizes): exclude; keep only stated design/nominal values.


In [1]:
import sys
sys.dont_write_bytecode = True
import asyncio
import json
import re
import sqlite3
import textwrap
from pathlib import Path

import pandas as pd
from IPython.display import display
from openai import AsyncOpenAI

# Find the repository from either notebooks/ or the repository root.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/polymer_samples.db").is_file())

BASE_URL = "http://localhost:8000/v1"
# MODEL = "qwen3.6-27b"
MODEL = "qwen3.8-27b"
THINKING = True  # Set False for thinking off.

# Skip a property extractor when property presence did not flag that property for the paper.
SKIP_ABSENT_PROPERTIES = True
# Print the model's reasoning trace (thinking mode) above each answer.
SHOW_REASONING = True


## Synthesis Prompts

In [2]:
SYNTH_SYSTEM_PROMPT = """
# Task

Extract the complete synthesis and preparation procedure for **one requested polymer/material sample** from the supplied paper.

Use the database record **only to identify the target material**. All procedural information must come from the paper, including its Methods, Results, tables, captions, schemes, footnotes, and supplementary text.

Return exactly:

`{"procedure_text": "..."}`

The value must be a single paragraph of past-tense scientific prose.

## 1. Identify the material and its full synthesis lineage

Identify the exact sample and its corresponding formulation, experimental run, or table entry.

Trace its preparation backward through all required precursors, including synthesized monomers, initiators, fillers, prepolymers, and parent polymers.

Describe the **complete in-paper preparation of each precursor before its use**, followed by the preparation of the target material.

- Use the target sample's specific quantities and conditions, not those of another sample or a representative experiment.
- Expand shared synthesis procedures using the target sample's own parameters.
- Include preparation details wherever they appear in the paper, not just in the Methods section.
- Do not reconstruct procedures that are only referenced through external publications.

If the target cannot be identified or no preparation is available, return `{"procedure_text": ""}`.

## 2. Preparation Details and Source Fidelity

Describe the complete preparation chronologically, from starting materials to the final material. Include all reported:

- **Materials:** identities, quantities, ratios, concentrations, purity, grade, molecular weight, particle size, physical form, and other specifications.
- **Operations:** reagent pretreatment, synthesis, purification, work-up, modification, formulation, blending, curing, and material processing (e.g., casting, molding, hot-pressing, annealing).
- **Conditions:** addition order, solvents, catalysts, atmosphere, temperature, time, pressure, equipment, washing, drying, and storage.

Include shaping when it produces the material, but exclude preparation solely for characterization or testing.

**Preserve the paper's information exactly.** Never invent, infer, calculate, convert, or silently correct procedural details. Retain original units, ranges, approximations, inequalities, concentration bases, and ratios. Associate each quantity with its correct material and step.

When the paper gives a representative or typical procedure plus a table, take the operations from the procedure but the amounts, ratios and times from the sample's own table row. Omit the representative run's specific amounts when they differ from that row.

**Examples:**

- **Combined loading:** "5 wt% CNT/graphene (1:2)" → Retain the combined loading and ratio; do not calculate individual percentages.
- **No inference:** "Heated at 80 °C with a condenser" ≠ "Refluxed at 80 °C."
- **Include processing:** "Melt-mixed at 180 °C for 10 min and compression-molded at 190 °C" → Retain both operations and their conditions.

## 3. Exclude results and non-preparation information

Exclude:

- **Measured outcomes:** yield, conversion, selectivity, isolated product mass, measured molecular weight, dispersity, polymerization degree, substitution degree, actual composition, and measured material properties.
- **Appearance and morphology:** descriptions such as white powder, yellow oil, transparent, viscous, or homogeneous.
- **Sample identifiers:** target sample codes, parent sample codes, compound numbers and series labels (e.g., "P20", "monomer 1", "poly3", "PGT-1"). Refer to every material, including synthesized monomers, initiators, prepolymers and parent polymers, by the chemical name or description the paper gives (e.g., "3-methyl-1,4-oxathiane-2,5-dione", "hydroxyl-terminated BHET oligomer"). If the paper gives no chemical name, describe the material by how it was made (e.g., "the cyclized product", "the amine-functionalized silica"). Never construct a chemical name that the paper does not state.
- **Testing:** characterization methods, ageing/degradation experiments, measurement thermal cycles, and test-specific specimen preparation.
- **Missing-information:** never mention unavailable conditions or externally referenced procedures.

Retain actual input amounts used in subsequent steps and explicitly stated target/theoretical recipe parameters.

**Example — remove outcomes but retain conditions:**

Paper: "The reaction was stirred at 60 °C for 12 h, yielding 2.5 g of a yellow polymer with 95% conversion."
Recipe: "The reaction was stirred at 60 °C for 12 h."

**Example — distinguish measured results from input amounts:**

Paper: "The polymer was obtained in 85% yield. Subsequently, 1.0 g of the polymer was dissolved in 20 mL of DMF."
Recipe: "The polymer (1.0 g) was dissolved in 20 mL of DMF."

**Example — remove labels but retain formulation:**

Paper: "Sample P20 was prepared by mixing 80 wt% PLA and 20 wt% PEG."
Recipe: "PLA (80 wt%) and PEG (20 wt%) were mixed."

## 4. Output and final verification

Write one continuous paragraph in the past tense, following the chronological preparation from starting materials through all required precursors to the final material.

Before returning, verify that:

- The correct sample, parent batches, and formulation were used.
- All available preparation stages and material-processing steps were included.
- Every reagent, quantity, operation, and condition is supported by the paper.
- No values were invented, derived, converted, or borrowed from another sample.
- No measured results, appearance descriptions, sample codes, or test procedures remain.

Return **only** the JSON object with the single field `procedure_text`. No headings, explanations, citations, or additional fields.
"""

SYNTH_USER_PROMPT = """
Extract the complete synthesis and preparation procedure for the requested sample, following the system instructions.

Return exactly `{"procedure_text": "..."}` with no additional fields or commentary.

Use the identification record only to identify the target material. Extract all procedural facts exclusively from the supplied paper.

Requested record: paper_uid={paper_uid}; sample_id={sample_id}.

<IDENTIFICATION_RECORD_DATA_ONLY>
{sample_record}
</IDENTIFICATION_RECORD_DATA_ONLY>

<PAPER_MARKDOWN_DATA_ONLY>
{markdown_full_text}
</PAPER_MARKDOWN_DATA_ONLY>
"""

SYNTH_RESPONSE_FORMAT = {'type': 'json_schema', 'json_schema': {'name': 'sample_synthesis_prose', 'strict': True, 'schema': {'type': 'object', 'additionalProperties': False, 'required': ['procedure_text'], 'properties': {'procedure_text': {'type': 'string'}}}}}


## Property Prompt


In [3]:
TG_SYSTEM_PROMPT = """
# Task

Extract every **glass transition temperature (Tg)** reported for **one requested polymer/material sample** from the supplied paper.

Use the database record **only to identify the target sample**. All values and measurement details must come from the paper, including its text, tables, captions, footnotes, and supplementary text.

Return exactly:

`{"properties": [...]}`

If the paper reports no Tg for this sample, return `{"properties": []}`.

## 1. Which values belong to the sample

Extract a Tg only if the paper clearly reports it for **this exact sample**.

- Do not extract values for other samples, precursors, components, or reference materials.
- Do not extract values quoted from other publications.
- Do not extract trends without a number (e.g., "Tg increased with filler content").
- Sample names can be shared by several samples, so identify the target from its whole record, especially the preparation or charecterization conditions it describes, and extract a value only when the paper reports it for a sample made under those same conditions.
- If it is unclear whether a value belongs to this sample, skip it.

One record per distinct reported value:

- The same value given in both text and table → one record.
- The same Tg measured by different methods, subtypes, heating rates, scans, or frequencies → separate records.

## 2. Fields

- **name:** always `glass_transition_temperature`.
- **method:** the technique used to measure this Tg: `DSC`, `DTA`, `DMA`, `TMA`, `dilatometry`, `dielectric`, `other`, or `unspecified` (if not stated). General instrument descriptions in the Methods section apply to this sample's measurement.
- **method_subtype:** which feature of the curve was taken as Tg. Use only if the paper states it; never infer it.
  - DSC or DTA: `onset`, `midpoint`, `inflection`, or `unspecified`.
  - DMA: `tan_delta_peak`, `loss_modulus_peak`, `storage_modulus_onset`, or `unspecified`.
  - All other methods: `null`.
  - A stated subtype not in these lists → `unspecified`, with the detail in `description`.
- **value_raw:** the value exactly as written, including ±, ranges, inequalities, and unit (e.g., `"152.3 ± 1.2 °C"`, `"120–125 °C"`, `"> 200 °C"`).
- **value_type** and **value:** classify the reported form and set a single number:

| value_type | Example | value |
|---|---|---|
| `exact` | `152 °C` | 152 |
| `mean_sd` | `152 ± 2 °C` | 152 |
| `approximate` | `~150 °C` | 150 |
| `range` | `120–125 °C` | 125 (upper end) |
| `lower_bound` | `> 200 °C` | 200 |
| `upper_bound` | `< −50 °C` | −50 |
| `not_observed` | `no Tg detected` | `null` |

  Use `not_observed` only when the paper explicitly states that no Tg was detected for this sample.

- **unit:** `°C`, `K`, or `°F`, as reported. Never convert.
- **conditions:** measurement conditions stated in the paper for this measurement. Each condition has:
  - `name`: `heating_rate`, `cooling_rate`, `scan` (e.g., `first_heating`, `second_heating`, `cooling`), `frequency`, `atmosphere`, `temperature_range`, `thermal_history` (treatment of the specimen just before measurement, e.g., quenching), `specimen_form` (e.g., film, powder), or `other`.
  - `type`: `exact`, `mean_sd`, `range`, `approximate`, `lower_bound`, or `upper_bound`.
  - `value`: a number when numeric (e.g., `10`), otherwise text (e.g., `"N2"`, `"second_heating"`).
  - `unit`: e.g., `"K/min"`, `"Hz"`, `"°C"`; `null` for non-numeric values.
  - `description`: `null`, or a short note (required when `name` is `other`).

  Include only conditions the paper states. Use an empty list if none.
- **origin:** `measured`, or `calculated` if the value comes from a model or equation (e.g., Fox, Gordon–Taylor, simulation).
- **description:** one short sentence saying what this value is (e.g., "Soft-segment Tg from the second DSC heating scan.").

## 3. Source fidelity

Never invent, infer, calculate, convert, or average values or conditions. Keep each value with its correct method and conditions. If a detail is not stated, use `unspecified`, `null`, or an empty list.

## 4. Output

Before returning, verify that:

- Every record is for the requested sample.
- No cited, plotted-only, or other-sample values are included.
- `value` follows the table above.
- `method_subtype` matches `method`.

Return **only** the JSON object. No explanations or extra fields.
"""

TG_USER_PROMPT = """
Extract every glass transition temperature reported for the requested sample, following the system instructions.

Return exactly `{"properties": [...]}`, or `{"properties": []}` if none is reported for this sample.

Use the identification record only to identify the target material. Extract all values and measurement details exclusively from the supplied paper.

Requested record: paper_uid={paper_uid}; sample_id={sample_id}.

<IDENTIFICATION_RECORD_DATA_ONLY>
{sample_record}
</IDENTIFICATION_RECORD_DATA_ONLY>

<PAPER_MARKDOWN_DATA_ONLY>
{markdown_full_text}
</PAPER_MARKDOWN_DATA_ONLY>
"""

TG_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "sample_tg",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "required": ["properties"],
            "properties": {
                "properties": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": [
                            "name", "method", "method_subtype", "value_raw", "value",
                            "value_type", "unit", "conditions", "origin", "description"
                        ],
                        "properties": {
                            "name": {"type": "string", "enum": ["glass_transition_temperature"]},
                            "method": {
                                "type": "string",
                                "enum": ["DSC", "DTA", "DMA", "TMA", "dilatometry", "dielectric", "other", "unspecified"]
                            },
                            "method_subtype": {
                                "type": ["string", "null"],
                                "enum": [
                                    "onset", "midpoint", "inflection",
                                    "tan_delta_peak", "loss_modulus_peak", "storage_modulus_onset",
                                    "unspecified", None
                                ]
                            },
                            "value_raw": {"type": "string"},
                            "value": {"type": ["number", "null"]},
                            "value_type": {
                                "type": "string",
                                "enum": ["exact", "mean_sd", "approximate", "range", "lower_bound", "upper_bound", "not_observed"]
                            },
                            "unit": {"type": "string", "enum": ["°C", "K", "°F"]},
                            "conditions": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "additionalProperties": False,
                                    "required": ["name", "type", "value", "unit", "description"],
                                    "properties": {
                                        "name": {
                                            "type": "string",
                                            "enum": [
                                                "heating_rate", "cooling_rate", "scan", "frequency", "atmosphere",
                                                "temperature_range", "thermal_history", "specimen_form", "other"
                                            ]
                                        },
                                        "type": {
                                            "type": "string",
                                            "enum": ["exact", "mean_sd", "approximate", "range", "lower_bound", "upper_bound"]
                                        },
                                        "value": {"type": ["number", "string"]},
                                        "unit": {"type": ["string", "null"]},
                                        "description": {"type": ["string", "null"]}
                                    }
                                }
                            },
                            "origin": {"type": "string", "enum": ["measured", "calculated"]},
                            "description": {"type": "string"}
                        }
                    }
                }
            }
        }
    }
}

## Property Prompt (per paper)

One request per paper instead of per sample. The model sees **all** samples of the paper, extracts every Tg value, and links each one to a sample (or to none) with `match_basis`. The response format is built per paper so `sample_id` can only be one of that paper's IDs.


In [4]:
TG_PAPER_SYSTEM_PROMPT = """
# Task

Extract every **glass transition temperature (Tg)** value reported in the supplied paper and link each value to a sample from the supplied sample list.

The sample list contains every sample recorded for this paper. Use it **only to identify samples**. All values and measurement details must come from the paper, including its text, tables, captions, footnotes, and supplementary text.

Return exactly:

`{"properties": [...]}`

If the paper reports no Tg for the target samples, return `{"properties": []}`.

## 1. Which values to report

Report every Tg value the paper gives for a target sample, and every Tg value that matches no sample in the list. Skip values that clearly belong to a listed sample that is not a target.

- Do not report values quoted from other publications.
- Do not read values off plots. Use only numbers stated in text, tables, or captions.
- Do not report trends without a number.

One record per distinct reported value:

- The same value given in both text and table → one record.
- The same Tg measured by different methods, subtypes, heating rates, scans, or frequencies → separate records.
- If the paper states that one value applies to several listed samples, report one record per sample.

## 2. Linking a value to a sample

For each value, first record what the paper says about the material, then choose the sample.

- **paper_label:** the name or label the paper uses for the material next to the value (e.g., the table row label).
- **identifying_data:** short text with the data reported together with the value that identify the material: table entry, preparation conditions, composition, molecular weight, dispersity. Use `""` if none.
- **match_basis:** how the value is tied to the chosen sample:
  - `same_entry`: the paper refers the value to the same table entry, run, or batch as the sample record.
  - `same_conditions`: the preparation conditions given with the value agree with the sample record.
  - `same_characterization`: the characterization data given with the value (molecular weight, dispersity, composition) agree with the sample record.
  - `name_only`: only the name or label matches; nothing else ties the value to this sample.
  - `none`: no listed sample matches.
- **sample_id:** the `sample_id` of the matched sample, or `null` when `match_basis` is `none`.

Rules:

- Several samples can share a name. A shared name alone never decides between them; compare the identifying data with each record.
- If the identifying data disagree with a sample record (e.g., a different molecular weight or different preparation conditions), the value does not belong to that sample.
- If no listed sample agrees with the identifying data, use `match_basis: none` and `sample_id: null`. This is a correct and common answer.

## 3. Value fields

- **name:** always `glass_transition_temperature`.
- **method:** the technique used to measure this Tg: `DSC`, `DTA`, `DMA`, `TMA`, `dilatometry`, `dielectric`, `other`, or `unspecified` (if not stated). Table footnotes and general instrument descriptions in the Methods section count as stated.
- **method_subtype:** which feature of the curve was taken as Tg. Use only if the paper states it; never infer it.
  - DSC or DTA: `onset`, `midpoint`, `inflection`, or `unspecified`.
  - DMA: `tan_delta_peak`, `loss_modulus_peak`, `storage_modulus_onset`, or `unspecified`.
  - All other methods: `null`.
- **value_raw:** the value exactly as written, including ±, ranges, inequalities, and unit.
- **value_type** and **value:**

| value_type | Example | value |
|---|---|---|
| `exact` | `152 °C` | 152 |
| `mean_sd` | `152 ± 2 °C` | 152 |
| `approximate` | `~150 °C` | 150 |
| `range` | `120–125 °C` | 125 (upper end) |
| `lower_bound` | `> 200 °C` | 200 |
| `upper_bound` | `< −50 °C` | −50 |
| `not_observed` | `no Tg detected` | `null` |

  Use `not_observed` only when the paper explicitly states that no Tg was detected.

- **unit:** `°C`, `K`, or `°F`, as reported. Never convert.
- **conditions:** measurement conditions stated in the paper for this measurement. Each condition has `name` (`heating_rate`, `cooling_rate`, `scan`, `frequency`, `atmosphere`, `temperature_range`, `thermal_history`, `specimen_form`, or `other`), `type`, `value` (number when numeric, otherwise text), `unit` (`null` for non-numeric values), and `description` (`null`, or a short note; required for `other`). Use an empty list if none are stated.
- **origin:** `measured`, or `calculated` if the value comes from a model or equation.

## 4. Source fidelity

Never invent, infer, calculate, convert, or average values or conditions. If a detail is not stated, use `unspecified`, `null`, `""`, or an empty list.

Return **only** the JSON object. No explanations or extra fields.
"""

TG_PAPER_USER_PROMPT = """
Extract every glass transition temperature reported in the paper and link each value to a sample from the sample list, following the system instructions.

Return exactly `{"properties": [...]}`.

paper_uid={paper_uid}
Target samples: {target_sample_ids}

<SAMPLE_LIST_DATA_ONLY>
{sample_list}
</SAMPLE_LIST_DATA_ONLY>

<PAPER_MARKDOWN_DATA_ONLY>
{markdown_full_text}
</PAPER_MARKDOWN_DATA_ONLY>
"""


def make_tg_paper_response_format(sample_ids):
    # Built per paper so sample_id can only be one of the paper's sample IDs (or null).
    # Linking fields come first so the model writes the evidence before it picks the sample.
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "paper_tg",
            "strict": True,
            "schema": {
                "type": "object",
                "additionalProperties": False,
                "required": ["properties"],
                "properties": {
                    "properties": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "additionalProperties": False,
                            "required": [
                                "paper_label", "value_raw", "identifying_data", "match_basis", "sample_id",
                                "name", "method", "method_subtype", "value_type", "value", "unit",
                                "conditions", "origin",
                            ],
                            "properties": {
                                "paper_label": {"type": "string"},
                                "value_raw": {"type": "string"},
                                "identifying_data": {"type": "string"},
                                "match_basis": {
                                    "type": "string",
                                    "enum": ["same_entry", "same_conditions", "same_characterization", "name_only", "none"],
                                },
                                "sample_id": {"type": ["integer", "null"], "enum": [int(i) for i in sample_ids] + [None]},
                                "name": {"type": "string", "enum": ["glass_transition_temperature"]},
                                "method": {
                                    "type": "string",
                                    "enum": ["DSC", "DTA", "DMA", "TMA", "dilatometry", "dielectric", "other", "unspecified"],
                                },
                                "method_subtype": {
                                    "type": ["string", "null"],
                                    "enum": ["onset", "midpoint", "inflection", "tan_delta_peak", "loss_modulus_peak",
                                             "storage_modulus_onset", "unspecified", None],
                                },
                                "value_type": {
                                    "type": "string",
                                    "enum": ["exact", "mean_sd", "approximate", "range", "lower_bound", "upper_bound", "not_observed"],
                                },
                                "value": {"type": ["number", "null"]},
                                "unit": {"type": "string", "enum": ["°C", "K", "°F"]},
                                "conditions": {
                                    "type": "array",
                                    "items": {
                                        "type": "object",
                                        "additionalProperties": False,
                                        "required": ["name", "type", "value", "unit", "description"],
                                        "properties": {
                                            "name": {
                                                "type": "string",
                                                "enum": ["heating_rate", "cooling_rate", "scan", "frequency", "atmosphere",
                                                         "temperature_range", "thermal_history", "specimen_form", "other"],
                                            },
                                            "type": {
                                                "type": "string",
                                                "enum": ["exact", "mean_sd", "approximate", "range", "lower_bound", "upper_bound"],
                                            },
                                            "value": {"type": ["number", "string"]},
                                            "unit": {"type": ["string", "null"]},
                                            "description": {"type": ["string", "null"]},
                                        },
                                    },
                                },
                                "origin": {"type": "string", "enum": ["measured", "calculated"]},
                            },
                        },
                    },
                },
            },
        },
    }


## Property Prompt (per sample, other samples as context)

The per-sample Tg prompt with one addition: the paper's other samples are listed **only to tell the samples apart**. The output format is unchanged (`TG_RESPONSE_FORMAT`), so results can be compared directly with the plain per-sample prompt and with the per-paper prompt. This prompt was copied from the per-sample prompt; edits to one are not carried over to the other.


In [5]:
TG_CTX_SYSTEM_PROMPT = """
# Task

Extract every **glass transition temperature (Tg)** reported for **one requested polymer/material sample** from the supplied paper.

Use the target record and the list of the paper's other samples **only to identify the target sample**. All values and measurement details must come from the paper, including its text, tables, captions, footnotes, and supplementary text.

Return exactly:

`{"properties": [...]}`

If the paper reports no Tg for this sample, return `{"properties": []}`.

## 1. Which values belong to the sample

Extract a Tg only if the paper clearly reports it for **this exact sample**.

- Do not extract values for other samples, precursors, components, or reference materials.
- Do not extract values quoted from other publications.
- Do not extract trends without a number (e.g., "Tg increased with filler content").
- Sample names can be shared by several samples, so identify the target from its whole record, especially the preparation or charecterization conditions it describes, and extract a value only when the paper reports it for a sample made under those same conditions.
- The paper's other samples are listed only so you can tell the samples apart. Compare the target record with them, and extract a value only if the paper reports it for the target and not for one of the other listed samples or for a material that is not in the list. Never extract values for the other samples.
- If it is unclear whether a value belongs to this sample, skip it.

One record per distinct reported value:

- The same value given in both text and table → one record.
- The same Tg measured by different methods, subtypes, heating rates, scans, or frequencies → separate records.

## 2. Fields

- **name:** always `glass_transition_temperature`.
- **method:** the technique used to measure this Tg: `DSC`, `DTA`, `DMA`, `TMA`, `dilatometry`, `dielectric`, `other`, or `unspecified` (if not stated). General instrument descriptions in the Methods section apply to this sample's measurement.
- **method_subtype:** which feature of the curve was taken as Tg. Use only if the paper states it; never infer it.
  - DSC or DTA: `onset`, `midpoint`, `inflection`, or `unspecified`.
  - DMA: `tan_delta_peak`, `loss_modulus_peak`, `storage_modulus_onset`, or `unspecified`.
  - All other methods: `null`.
  - A stated subtype not in these lists → `unspecified`, with the detail in `description`.
- **value_raw:** the value exactly as written, including ±, ranges, inequalities, and unit (e.g., `"152.3 ± 1.2 °C"`, `"120–125 °C"`, `"> 200 °C"`).
- **value_type** and **value:** classify the reported form and set a single number:

| value_type | Example | value |
|---|---|---|
| `exact` | `152 °C` | 152 |
| `mean_sd` | `152 ± 2 °C` | 152 |
| `approximate` | `~150 °C` | 150 |
| `range` | `120–125 °C` | 125 (upper end) |
| `lower_bound` | `> 200 °C` | 200 |
| `upper_bound` | `< −50 °C` | −50 |
| `not_observed` | `no Tg detected` | `null` |

  Use `not_observed` only when the paper explicitly states that no Tg was detected for this sample.

- **unit:** `°C`, `K`, or `°F`, as reported. Never convert.
- **conditions:** measurement conditions stated in the paper for this measurement. Each condition has:
  - `name`: `heating_rate`, `cooling_rate`, `scan` (e.g., `first_heating`, `second_heating`, `cooling`), `frequency`, `atmosphere`, `temperature_range`, `thermal_history` (treatment of the specimen just before measurement, e.g., quenching), `specimen_form` (e.g., film, powder), or `other`.
  - `type`: `exact`, `mean_sd`, `range`, `approximate`, `lower_bound`, or `upper_bound`.
  - `value`: a number when numeric (e.g., `10`), otherwise text (e.g., `"N2"`, `"second_heating"`).
  - `unit`: e.g., `"K/min"`, `"Hz"`, `"°C"`; `null` for non-numeric values.
  - `description`: `null`, or a short note (required when `name` is `other`).

  Include only conditions the paper states. Use an empty list if none.
- **origin:** `measured`, or `calculated` if the value comes from a model or equation (e.g., Fox, Gordon–Taylor, simulation).
- **description:** one short sentence saying what this value is (e.g., "Soft-segment Tg from the second DSC heating scan.").

## 3. Source fidelity

Never invent, infer, calculate, convert, or average values or conditions. Keep each value with its correct method and conditions. If a detail is not stated, use `unspecified`, `null`, or an empty list.

## 4. Output

Before returning, verify that:

- Every record is for the requested sample.
- No cited, plotted-only, or other-sample values are included.
- `value` follows the table above.
- `method_subtype` matches `method`.

Return **only** the JSON object. No explanations or extra fields.
"""

TG_CTX_USER_PROMPT = """
Extract every glass transition temperature reported for the requested sample, following the system instructions.

Return exactly `{"properties": [...]}`, or `{"properties": []}` if none is reported for this sample.

Use the target record and the other samples only to identify the target material. Extract all values and measurement details exclusively from the supplied paper.

Requested record: paper_uid={paper_uid}; sample_id={sample_id}.

<IDENTIFICATION_RECORD_DATA_ONLY>
{sample_record}
</IDENTIFICATION_RECORD_DATA_ONLY>

Other samples in this paper, for telling samples apart only (do not extract values for them):

<OTHER_SAMPLES_DATA_ONLY>
{other_samples}
</OTHER_SAMPLES_DATA_ONLY>

<PAPER_MARKDOWN_DATA_ONLY>
{markdown_full_text}
</PAPER_MARKDOWN_DATA_ONLY>
"""

# Output format is the same as the per-sample prompt: TG_RESPONSE_FORMAT.


## Property extractors and helpers

`PROPERTY_EXTRACTORS` maps a property-presence label (as stored in `property_classification.properties_json`) to its prompts. To add Td later, add one entry here and one check function. `ask_qwen` and `parse_response` are shared by both send cells.


In [6]:
PROPERTY_EXTRACTORS = {
    "glass transition temperature": {
        "key": "tg",
        "system": TG_SYSTEM_PROMPT,
        "user": TG_USER_PROMPT,
        "response_format": TG_RESPONSE_FORMAT,
    },
}

# Rules strict JSON schema cannot express; checked after parsing.
TG_SUBTYPES = {
    "DSC": {"onset", "midpoint", "inflection", "unspecified"},
    "DTA": {"onset", "midpoint", "inflection", "unspecified"},
    "DMA": {"tan_delta_peak", "loss_modulus_peak", "storage_modulus_onset", "unspecified"},
}


def check_tg_record(record):
    problems = []
    if (record.get("value") is None) != (record.get("value_type") == "not_observed"):
        problems.append("value must be null exactly when value_type is not_observed")
    allowed = TG_SUBTYPES.get(record.get("method"))
    subtype = record.get("method_subtype")
    if allowed is None and subtype is not None:
        problems.append(f"method_subtype must be null for method {record.get('method')}")
    if allowed is not None and subtype not in allowed:
        problems.append(f"method_subtype {subtype!r} not allowed for {record.get('method')}")
    return "; ".join(problems)


RECORD_CHECKS = {"tg": check_tg_record}


def format_conditions(conditions):
    parts = []
    for condition in conditions or []:
        text = f"{condition.get('name')}={condition.get('value')}"
        if condition.get("unit"):
            text += f" {condition['unit']}"
        if condition.get("type") not in (None, "exact"):
            text += f" ({condition['type']})"
        parts.append(text)
    return "; ".join(parts)


async def ask_qwen(client, messages, response_format):
    return await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
        max_tokens=20_000,
        extra_body={"chat_template_kwargs": {"enable_thinking": THINKING}},
        response_format=response_format,
    )


def get_reasoning(response):
    # vLLM returns the thinking trace as `reasoning_content` (older) or `reasoning` (newer).
    if isinstance(response, Exception):
        return ""
    message = response.choices[0].message
    reasoning = getattr(message, "reasoning_content", None) or getattr(message, "reasoning", None)
    if not reasoning and getattr(message, "model_extra", None):
        reasoning = message.model_extra.get("reasoning_content") or message.model_extra.get("reasoning")
    if not reasoning:
        # Fallback when the server leaves <think>...</think> inside the content.
        match = re.search(r"<think>(.*?)</think>", message.content or "", flags=re.S)
        reasoning = match.group(1) if match else ""
    return (reasoning or "").strip()


def print_reasoning(response):
    if not SHOW_REASONING:
        return
    reasoning = get_reasoning(response)
    print("--- Reasoning ---")
    print(reasoning or "[No reasoning returned; is THINKING on?]")
    print("--- Answer ---")


def parse_response(response):
    # Returns (parsed JSON or None, note); note is empty when everything went fine.
    if isinstance(response, Exception):
        return None, f"request failed: {response!r}"
    notes = []
    if response.choices[0].finish_reason != "stop":
        notes.append(f"Response may be incomplete. Finish reason: {response.choices[0].finish_reason}")
    answer = response.choices[0].message.content or ""
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()
    try:
        return json.loads(answer), "\n".join(notes)
    except json.JSONDecodeError:
        notes.append(answer or "[No answer content returned]")
        return None, "\n".join(notes)


## List samples and property presence for a paper

Change `PAPER_UID` and run this cell. Every sample record for that paper is shown, followed by the properties the property-presence classifier found. Property presence is a **paper-level** label: it says the paper reports the property for at least one sample, not that this particular sample has it.


In [7]:
PAPER_UID = "ID000732356"

In [8]:
def read_db(query, params):
    connection = sqlite3.connect(
        f"file:{ROOT / 'data/polymer_samples.db'}?mode=ro", uri=True
    )
    try:
        return pd.read_sql_query(query, connection, params=params)
    finally:
        connection.close()


def get_samples(paper_uid):
    return read_db(
        "SELECT s.paper_uid, p.canonical_source AS source, s.sample_id, "
        "s.name, s.identifier, s.aliases, s.description "
        "FROM samples s JOIN papers p USING (paper_uid) "
        "WHERE s.paper_uid = ? ORDER BY s.sample_id",
        (paper_uid,),
    )


def get_paper_title(paper_uid):
    rows = read_db("SELECT title FROM papers WHERE paper_uid = ?", (paper_uid,))
    return rows.title.iloc[0] if not rows.empty and rows.title.iloc[0] else "[no title in database]"


def get_property_presence(paper_uid):
    # Returns (classification status, list of lower-case property names) for the paper.
    rows = read_db(
        "SELECT status, properties_json FROM property_classification WHERE paper_uid = ?",
        (paper_uid,),
    )
    if rows.empty:
        return "not classified", []
    status = rows.status.iloc[0]
    if status != "success":
        return status, []
    return status, [name.strip().lower() for name in json.loads(rows.properties_json.iloc[0])]


def show_property_presence(paper_uid):
    status, present = get_property_presence(paper_uid)
    print(f"Property presence for {paper_uid} ({status}): " + (", ".join(present) if present else "none"))
    for label, extractor in PROPERTY_EXTRACTORS.items():
        flagged = label in present
        will_run = flagged or not SKIP_ABSENT_PROPERTIES
        print(f"  {extractor['key']}: {'flagged' if flagged else 'not flagged'}; "
              f"extractor {'will run' if will_run else 'skipped'}")


samples = get_samples(PAPER_UID)
print(f"{PAPER_UID}: {get_paper_title(PAPER_UID)}")
print(f"{len(samples)} samples")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(samples)
show_property_presence(PAPER_UID)


ID000732356: Ring-opening polymerization of six-membered cyclic hybrid dimers composed of an oxoester and thioester
29 samples


,paper_uid,source,sample_id,name,identifier,aliases,description
0,ID000732356,springer_nature,1751756,poly1,Table 1 Entry 1,[],"Linear polymer composed of alternating oxoester-thioester units derived from monomer 1 (3-methyl-1,4-oxathiane-2,5-dione). Synthesized via anionic ring-opening polymerization in CH2Cl2 at room temperature using 2-ethylhexyl thioglycolate (5 mol%) as initiator and pyridine (5 mol%) as catalyst for 5 days. Conversion: 14%, Mn: 690, PDI: 1.16."
1,ID000732356,springer_nature,1751757,poly2,Table 1 Entry 2,[],"Linear polymer composed of alternating oxoester-thioester units derived from monomer 2 (6-methyl-1,4-oxathiane-2,5-dione). Synthesized via anionic ring-opening polymerization in CH2Cl2 at room temperature using 2-ethylhexyl thioglycolate (5 mol%) as initiator and pyridine (5 mol%) as catalyst for 5 days. Conversion: 23%, Mn: 800, PDI: 1.32."
2,ID000732356,springer_nature,1751758,poly3,Table 1 Entry 3,[],"Linear polymer composed of alternating oxoester-thioester units derived from monomer 3 (3,6-dimethyl-1,4-oxathiane-2,5-dione). Synthesized via anionic ring-opening polymerization in CH2Cl2 at room temperature using 2-ethylhexyl thioglycolate (5 mol%) as initiator and pyridine (5 mol%) as catalyst for 5 days. Conversion: 10%, Mn: 600, PDI: 1.18."
3,ID000732356,springer_nature,1751759,poly1,Table 2 Entry 1,[],"Linear polymer composed of alternating oxoester-thioester units derived from monomer 1. Synthesized via anionic ring-opening polymerization in CH2Cl2 at room temperature using 2-ethylhexyl thioglycolate (6 mol%) as initiator and 2,6-lutidine (5 mol%) as catalyst. Sampled at 1 day (28% conversion, Mn 1050, PDI 1.13) and 3 days (51% conversion, Mn 1900, PDI 1.12)."
4,ID000732356,springer_nature,1751760,poly1,Table 2 Entry 2,[],"Linear polymer composed of alternating oxoester-thioester units derived from monomer 1. Synthesized via anionic ring-opening polymerization in CH2Cl2 at room temperature using 2-ethylhexyl thioglycolate (6 mol%) as initiator and 2,6-lutidine (10 mol%) as catalyst for 1 day. Conversion: 71%, Mn: 2700, PDI: 1.13."
5,ID000732356,springer_nature,1751761,poly1,Table 2 Entry 3,[],"Linear polymer composed of alternating oxoester-thioester units derived from monomer 1. Synthesized via anionic ring-opening polymerization in CH2Cl2 at room temperature using 2-ethylhexyl thioglycolate (5 mol%) as initiator and 2,6-lutidine (10 mol%) as catalyst for 5 days. Conversion: 87%, Mn: 3100, PDI: 1.33."
6,ID000732356,springer_nature,1751762,poly1,Table 2 Entry 4,[],"Linear polymer composed of alternating oxoester-thioester units derived from monomer 1. Synthesized via anionic ring-opening polymerization in CH2Cl2 at room temperature using 2-ethylhexyl thioglycolate (2 mol%) as initiator and 2,6-lutidine (10 mol%) as catalyst for 2 days. Conversion: 51%, Mn: 4600, PDI: 1.12."
7,ID000732356,springer_nature,1751763,poly2,Table 2 Entry 5,[],"Linear polymer composed of alternating oxoester-thioester units derived from monomer 2. Synthesized via anionic ring-opening polymerization in CH2Cl2 at room temperature using 2-ethylhexyl thioglycolate (6 mol%) as initiator and 2,6-lutidine (5 mol%) as catalyst. Sampled at 1 day (71% conversion, Mn 1800, PDI 1.23) and 2 days (84% conversion, Mn 2400, PDI 1.26)."
8,ID000732356,springer_nature,1751764,poly2,Table 2 Entry 6,[],"Linear polymer composed of alternating oxoester-thioester units derived from monomer 2. Synthesized via anionic ring-opening polymerization in CH2Cl2 at room temperature using 2-ethylhexyl thioglycolate (6 mol%) as initiator and 2,6-lutidine (10 mol%) as catalyst for 1 day. Conversion: 83%, Mn: 2900, PDI: 1.17."
9,ID000732356,springer_nature,1751765,poly2,Table 2 Entry 7,[],"Linear polymer composed of alternating oxoester-thioester units derived from monomer 2. Synthesized via anionic ring-opening polymerization in CH2Cl2 at room temperature using 2-ethylhexyl thioglycolate (5 mol%) as initiator and 2,6-lutidine (10 mol%) as catalyst for 5 

Property presence for ID000732356 (success): number-average molecular weight, dispersity, glass transition temperature, thermal decomposition temperature
  tg: flagged; extractor will run


## Choose a sample and load the full text

Set both IDs here. This cell reads the selected database record and full local Markdown, then builds `synthesis_messages` and one property request per flagged property in `property_jobs`. It does not call Qwen. You can inspect `sample_record`, `paper_text`, `synthesis_messages` or `property_jobs` before sending.


In [9]:
SAMPLE_ID = 1751759

In [10]:
# Fetch again so the chosen UID need not match the preceding display cell.
paper_samples = get_samples(PAPER_UID)
selected = paper_samples.loc[paper_samples.sample_id == SAMPLE_ID]
if len(selected) != 1:
    raise ValueError(f"Sample {SAMPLE_ID} does not belong to paper {PAPER_UID}.")

row = selected.iloc[0]
md_path = ROOT / "data/fulltext" / row["source"] / PAPER_UID / f"{PAPER_UID}.md"
paper_text = md_path.read_text(encoding="utf-8")
if not paper_text.strip():
    raise ValueError(f"The paper Markdown is empty: {md_path}")
sample_record = {key: row[key] for key in ("name", "identifier", "aliases", "description")}

values = {
    "paper_uid": PAPER_UID,
    "sample_id": str(SAMPLE_ID),
    "sample_record": json.dumps(sample_record, ensure_ascii=False),
    "markdown_full_text": paper_text,
}


def build_messages(system_prompt, user_template):
    user_message = re.sub(
        r"\{(paper_uid|sample_id|sample_record|markdown_full_text)\}",
        lambda match: values[match.group(1)], user_template,
    )
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]


synthesis_messages = build_messages(SYNTH_SYSTEM_PROMPT, SYNTH_USER_PROMPT)

# Property key (e.g. "tg") -> (messages, response_format), only for properties flagged for this paper.
presence_status, present_properties = get_property_presence(PAPER_UID)
property_jobs = {}
for label, extractor in PROPERTY_EXTRACTORS.items():
    if label in present_properties or not SKIP_ABSENT_PROPERTIES:
        property_jobs[extractor["key"]] = (
            build_messages(extractor["system"], extractor["user"]),
            extractor["response_format"],
        )

display(selected)
print(f"Loaded full Markdown: {len(paper_text):,} characters")
show_property_presence(PAPER_UID)
print("Property requests to send:", ", ".join(property_jobs) or "none")


,paper_uid,source,sample_id,name,identifier,aliases,description
3,ID000732356,springer_nature,1751759,poly1,Table 2 Entry 1,[],Linear polymer composed of alternating oxoeste...


Loaded full Markdown: 36,620 characters
Property presence for ID000732356 (success): number-average molecular weight, dispersity, glass transition temperature, thermal decomposition temperature
  tg: flagged; extractor will run
Property requests to send: tg


## Send synthesis to Qwen

One model request per execution. The full response stays in `synthesis_response`; the paragraph is `procedure_text` and the reasoning trace is `synthesis_reasoning` (printed above the answer when `SHOW_REASONING` is on). Nothing is written to disk.


In [11]:
# async with AsyncOpenAI(
#     base_url=BASE_URL, api_key="none", timeout=1200, max_retries=0
# ) as client:
#     try:
#         synthesis_response = await ask_qwen(client, synthesis_messages, SYNTH_RESPONSE_FORMAT)
#     except Exception as error:
#         synthesis_response = error

# procedure_text = None
# synthesis_reasoning = get_reasoning(synthesis_response)
# print_reasoning(synthesis_response)
# parsed, note = parse_response(synthesis_response)
# if isinstance(parsed, dict) and isinstance(parsed.get("procedure_text"), str):
#     procedure_text = parsed["procedure_text"]
#     print(textwrap.fill(procedure_text or "[Empty synthesis returned]", width=100))
# if note:
#     print(note)


## Send properties to Qwen

Sends one request per property in `property_jobs`, all at the same time. Full responses stay in `property_responses`; records are in `property_records`, e.g. `property_records["tg"]`, and reasoning traces are in `property_reasoning` (printed above each table when `SHOW_REASONING` is on). The `check` column flags records that break the rules strict JSON schema cannot enforce. Nothing is written to disk.


In [12]:
async with AsyncOpenAI(
    base_url=BASE_URL, api_key="none", timeout=1200, max_retries=0
) as client:
    results = await asyncio.gather(
        *(ask_qwen(client, messages, response_format)
          for messages, response_format in property_jobs.values()),
        return_exceptions=True,
    )
property_responses = dict(zip(property_jobs, results))

property_records = {}
property_reasoning = {key: get_reasoning(response) for key, response in property_responses.items()}
for label, extractor in PROPERTY_EXTRACTORS.items():
    key = extractor["key"]
    print(f"\n=== {label} ({key}) ===")
    if key not in property_responses:
        print("Skipped: not flagged by property presence for this paper.")
        continue
    print_reasoning(property_responses[key])
    parsed, note = parse_response(property_responses[key])
    records = parsed.get("properties") if isinstance(parsed, dict) else None
    if records is None:
        print(note or "[No parsable answer]")
        continue
    property_records[key] = records
    if not records:
        print("No values reported for this sample.")
    else:
        check = RECORD_CHECKS.get(key, lambda record: "")
        table = pd.DataFrame([
            {
                "method": r.get("method"),
                "method_subtype": r.get("method_subtype"),
                "value_raw": r.get("value_raw"),
                "value": r.get("value"),
                "value_type": r.get("value_type"),
                "unit": r.get("unit"),
                "origin": r.get("origin"),
                "conditions": format_conditions(r.get("conditions")),
                "description": r.get("description"),
                "check": check(r),
            }
            for r in records
        ])
        with pd.option_context("display.max_colwidth", None):
            display(table)
    if note:
        print(note)



=== glass transition temperature (tg) ===
--- Reasoning ---
We need answer user's request: extract Tg for requested sample from paper. Need output JSON only. Need analyze in English (same language as user? user English). Need identify target sample: paper_uid ID000732356; sample_id 1751759. Identification record: name poly1, identifier Table 2 Entry 1, description: Linear polymer composed of alternating oxoester-thioester units derived from monomer 1. Synthesized via anionic ROP in CH2Cl2 at room temp using 2-ethylhexyl thioglycolate (6 mol%) as initiator and 2,6-lutidine (5 mol%) as catalyst. Sampled at 1 day (28% conversion, Mn 1050, PDI 1.13) and 3 days (51% conversion, Mn 1900, PDI 1.12).

Need extract every Tg reported for this exact sample from paper. Paper has Table 6 thermal properties: poly1 Mn 3300 PDI 1.82 Tg 3.7 °C, Td10 194. But target sample is Table 2 Entry 1: monomer 1, init 6 mol%, lutidine 5 mol%, time 1 day conv 28 Mn 1050 PDI 1.13; 3 days conv 51 Mn 1900 PDI 1.12. 

## Send properties per paper to Qwen

Needs only `PAPER_UID` (no `SAMPLE_ID`). Every request lists all samples of the paper; papers with more than `TARGET_BATCH_SIZE` samples are split into several requests that differ only in which samples are targets. Messages are built in this cell from the current prompt, so prompt edits take effect without re-running other cells.

Code decides which values to keep: `sample_id` set and `match_basis` is `same_entry`, `same_conditions` or `same_characterization`; `name_only` is kept only when no other sample in the paper has the same name. All values, kept or not, are in `paper_tg_records`; reasoning traces are in `paper_tg_reasoning`. Nothing is written to disk.


In [13]:
# Samples per request. Every request lists all samples of the paper; only the targets change.
TARGET_BATCH_SIZE = 15
STRONG_MATCHES = {"same_entry", "same_conditions", "same_characterization"}

paper_samples_all = get_samples(PAPER_UID)
if paper_samples_all.empty:
    raise ValueError(f"No samples for {PAPER_UID}.")
paper_md_path = ROOT / "data/fulltext" / paper_samples_all.source.iloc[0] / PAPER_UID / f"{PAPER_UID}.md"
paper_markdown = paper_md_path.read_text(encoding="utf-8")

all_ids = [int(i) for i in paper_samples_all.sample_id]
sample_list = "\n".join(
    json.dumps({
        "sample_id": int(r.sample_id), "name": r.name, "identifier": r.identifier,
        "aliases": json.loads(r.aliases), "description": r.description,
    }, ensure_ascii=False)
    for r in paper_samples_all.itertuples()
)
batches = [all_ids[i:i + TARGET_BATCH_SIZE] for i in range(0, len(all_ids), TARGET_BATCH_SIZE)]
tg_paper_format = make_tg_paper_response_format(all_ids)


def build_paper_messages(targets):
    values = {
        "paper_uid": PAPER_UID,
        "target_sample_ids": "all samples in the list" if len(batches) == 1 else ", ".join(map(str, targets)),
        "sample_list": sample_list,
        "markdown_full_text": paper_markdown,
    }
    user = re.sub(r"\{(paper_uid|target_sample_ids|sample_list|markdown_full_text)\}",
                  lambda m: values[m.group(1)], TG_PAPER_USER_PROMPT)
    return [{"role": "system", "content": TG_PAPER_SYSTEM_PROMPT}, {"role": "user", "content": user}]


print(f"{PAPER_UID}: {get_paper_title(PAPER_UID)}")
print(f"{len(all_ids)} samples, {len(batches)} request(s)")

# Messages are built here, from the current prompt cell, so prompt edits always take effect.
async with AsyncOpenAI(base_url=BASE_URL, api_key="none", timeout=1200, max_retries=0) as client:
    paper_tg_responses = await asyncio.gather(
        *(ask_qwen(client, build_paper_messages(targets), tg_paper_format) for targets in batches),
        return_exceptions=True,
    )
paper_tg_reasoning = [get_reasoning(r) for r in paper_tg_responses]

# Collect records; keep only those for the batch's own targets (or unmatched ones).
names = paper_samples_all.set_index("sample_id")
name_counts = paper_samples_all.name.str.strip().str.lower().value_counts()
rows, seen_unmatched = [], set()
for b, (targets, response) in enumerate(zip(batches, paper_tg_responses), 1):
    print(f"\n=== Request {b}/{len(batches)} ===")
    print_reasoning(response)
    parsed, note = parse_response(response)
    if note:
        print(note)
    for r in (parsed or {}).get("properties", []):
        sid = r.get("sample_id")
        if sid is not None and sid not in targets:
            continue  # belongs to another batch's targets
        if sid is None:
            key = (r.get("paper_label"), r.get("value_raw"), r.get("method"))
            if key in seen_unmatched:
                continue
            seen_unmatched.add(key)
        problems = [p for p in [check_tg_record(r)] if p]
        if (r.get("match_basis") == "none") != (sid is None):
            problems.append("match_basis none must go with sample_id null")
        name = names["name"].get(sid, "") if sid is not None else ""
        unique_name = sid is not None and name_counts.get(name.strip().lower(), 0) == 1
        basis = r.get("match_basis")
        kept = sid is not None and (basis in STRONG_MATCHES or (basis == "name_only" and unique_name))
        rows.append({
            "kept": kept,
            "sample_id": sid,
            "sample": f"{name} | {names['identifier'].get(sid, '')}" if sid is not None else "",
            "paper_label": r.get("paper_label"),
            "identifying_data": r.get("identifying_data"),
            "match_basis": basis,
            "value_raw": r.get("value_raw"),
            "value": r.get("value"),
            "value_type": r.get("value_type"),
            "method": r.get("method"),
            "method_subtype": r.get("method_subtype"),
            "conditions": format_conditions(r.get("conditions")),
            "origin": r.get("origin"),
            "check": "; ".join(problems),
        })

paper_tg_records = pd.DataFrame(rows)
if paper_tg_records.empty:
    print("\nNo Tg values returned for this paper.")
else:
    # Same value assigned to more than one sample.
    kept_rows = paper_tg_records[paper_tg_records.kept]
    dup = kept_rows.groupby(["value_raw", "method", "conditions"]).sample_id.transform("nunique") > 1
    idx = kept_rows.index[dup.to_numpy()]
    paper_tg_records.loc[idx, "check"] = (paper_tg_records.loc[idx, "check"] + "; assigned to several samples").str.strip("; ")
    print(f"\nAll Tg values ({len(paper_tg_records)}; kept {int(paper_tg_records.kept.sum())}):")
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(paper_tg_records)

    summary = paper_samples_all[["sample_id", "name", "identifier"]].copy()
    kept_values = (paper_tg_records[paper_tg_records.kept]
                   .groupby("sample_id").value_raw.apply(lambda v: "; ".join(v)))
    summary["kept_tg"] = summary.sample_id.map(kept_values).fillna("")
    print("\nKept Tg per sample:")
    with pd.option_context("display.max_rows", None):
        display(summary)


ID000732356: Ring-opening polymerization of six-membered cyclic hybrid dimers composed of an oxoester and thioester
29 samples, 2 request(s)

=== Request 1/2 ===
--- Reasoning ---
Let me analyze the paper to find all glass transition temperature (Tg) values reported.

Looking at the paper, I can find Tg values in Table 6:

**Table 6. Thermal properties of the polymers**

| | Mn | PDI | Tg (°C)^a | Td10 (°C)^b |
|---|---|---|---|---|
| poly(rac-thiolactide)^c | 3860 | 1.5 | −1 ~ 0 | 162 |
| poly(rac-lactide) | – | – | 35.5 ~ 65^d | 250^f |
| poly1 | 3300 | 1.82 | 3.7 | 194 |
| poly2 | 6100 | 1.15 | 16.2 | 209 |
| poly3 | 3400 | 1.56 | 1.4 | 249 |

Now let me identify the Tg values:

1. **poly(rac-thiolactide)**: Tg = −1 ~ 0 °C, Mn = 3860, PDI = 1.5
2. **poly(rac-lactide)**: Tg = 35.5 ~ 65 °C (with footnote d)
3. **poly1**: Tg = 3.7 °C, Mn = 3300, PDI = 1.82
4. **poly2**: Tg = 16.2 °C, Mn = 6100, PDI = 1.15
5. **poly3**: Tg = 1.4 °C, Mn = 3400, PDI = 1.56

Now let me check the footnotes.

,kept,sample_id,sample,paper_label,identifying_data,match_basis,value_raw,value,value_type,method,method_subtype,conditions,origin,check
0,False,None,,poly(rac-thiolactide),"Mn = 3860, PDI = 1.5",none,−1 ~ 0,0.0,range,unspecified,None,,measured,
1,False,None,,poly(rac-lactide),,none,35.5 ~ 65,65.0,range,unspecified,None,,measured,
2,False,None,,poly1,"Mn = 3300, PDI = 1.82",none,3.7,3.7,exact,unspecified,None,,measured,
3,False,None,,poly2,"Mn = 6100, PDI = 1.15",none,16.2,16.2,exact,unspecified,None,,measured,
4,False,None,,poly3,"Mn = 3400, PDI = 1.56",none,1.4,1.4,exact,unspecified,None,,measured,
5,False,None,,poly(rac-thiolactide),"Mn 3860, PDI 1.5",none,−1 ~ 0 °C,0.0,range,unspecified,None,,measured,
6,False,None,,poly(rac-lactide),,none,35.5 ~ 65 °C,65.0,range,unspecified,None,,measured,
7,False,None,,poly1,"Mn 3300, PDI 1.82",none,3.7 °C,3.7,exact,unspecified,None,,measured,
8,False,None,,poly3,"Mn 3400, PDI 1.56",none,1.4 °C,1.4,exact,unspecified,None,,measured,



Kept Tg per sample:


,sample_id,name,identifier,kept_tg
0,1751756,poly1,Table 1 Entry 1,
1,1751757,poly2,Table 1 Entry 2,
2,1751758,poly3,Table 1 Entry 3,
3,1751759,poly1,Table 2 Entry 1,
4,1751760,poly1,Table 2 Entry 2,
5,1751761,poly1,Table 2 Entry 3,
6,1751762,poly1,Table 2 Entry 4,
7,1751763,poly2,Table 2 Entry 5,
8,1751764,poly2,Table 2 Entry 6,
9,1751765,poly2,Table 2 Entry 7,


## Send properties per sample with other samples as context

Uses `PAPER_UID` and `SAMPLE_ID`. With `CTX_RUN_ALL_SAMPLES = True` it runs every sample of the paper (one request each, `CTX_MAX_CONCURRENT` at a time) and shows the Tg per sample, for comparison with the per-paper run. Messages are built in this cell from the current prompt. Records are in `ctx_tg_records`, reasoning in `ctx_tg_reasoning`. A value returned for more than one sample is flagged in `check`. Nothing is written to disk.


In [14]:
# Per-sample Tg extraction, with the paper's other samples listed for identification only.
# Set CTX_RUN_ALL_SAMPLES = True to run every sample of PAPER_UID (one request each) and compare
# with the per-paper run; otherwise only SAMPLE_ID is run.
CTX_RUN_ALL_SAMPLES = False
CTX_MAX_CONCURRENT = 4

ctx_samples = get_samples(PAPER_UID)
if ctx_samples.empty:
    raise ValueError(f"No samples for {PAPER_UID}.")
ctx_md_path = ROOT / "data/fulltext" / ctx_samples.source.iloc[0] / PAPER_UID / f"{PAPER_UID}.md"
ctx_markdown = ctx_md_path.read_text(encoding="utf-8")


def sample_json(r):
    return json.dumps({
        "sample_id": int(r.sample_id), "name": r.name, "identifier": r.identifier,
        "aliases": json.loads(r.aliases), "description": r.description,
    }, ensure_ascii=False)


def build_ctx_messages(sample_id):
    target = ctx_samples.loc[ctx_samples.sample_id == sample_id]
    if len(target) != 1:
        raise ValueError(f"Sample {sample_id} does not belong to paper {PAPER_UID}.")
    t = target.iloc[0]
    others = ctx_samples.loc[ctx_samples.sample_id != sample_id]
    values = {
        "paper_uid": PAPER_UID,
        "sample_id": str(sample_id),
        "sample_record": json.dumps({k: t[k] for k in ("name", "identifier", "aliases", "description")},
                                    ensure_ascii=False),
        "other_samples": "\n".join(sample_json(r) for r in others.itertuples()) or "[none]",
        "markdown_full_text": ctx_markdown,
    }
    user = re.sub(r"\{(paper_uid|sample_id|sample_record|other_samples|markdown_full_text)\}",
                  lambda m: values[m.group(1)], TG_CTX_USER_PROMPT)
    return [{"role": "system", "content": TG_CTX_SYSTEM_PROMPT}, {"role": "user", "content": user}]


ctx_ids = [int(i) for i in ctx_samples.sample_id] if CTX_RUN_ALL_SAMPLES else [int(SAMPLE_ID)]
print(f"{PAPER_UID}: {get_paper_title(PAPER_UID)}")
print(f"{len(ctx_samples)} samples in paper; running {len(ctx_ids)} request(s)")

# Messages are built here, from the current prompt cell, so prompt edits always take effect.
semaphore = asyncio.Semaphore(CTX_MAX_CONCURRENT)
async with AsyncOpenAI(base_url=BASE_URL, api_key="none", timeout=1200, max_retries=0) as client:
    async def run_one(sample_id):
        async with semaphore:
            return await ask_qwen(client, build_ctx_messages(sample_id), TG_RESPONSE_FORMAT)
    ctx_responses = dict(zip(ctx_ids, await asyncio.gather(*(run_one(i) for i in ctx_ids),
                                                             return_exceptions=True)))
ctx_tg_reasoning = {i: get_reasoning(r) for i, r in ctx_responses.items()}

names = ctx_samples.set_index("sample_id")
rows = []
for sample_id, response in ctx_responses.items():
    print(f"\n=== {sample_id}: {names['name'][sample_id]} | {names['identifier'][sample_id]} ===")
    print_reasoning(response)
    parsed, note = parse_response(response)
    records = parsed.get("properties") if isinstance(parsed, dict) else None
    if records is None:
        print(note or "[No parsable answer]")
        continue
    if not records:
        print("No values reported for this sample.")
    for r in records:
        rows.append({
            "sample_id": sample_id,
            "sample": f"{names['name'][sample_id]} | {names['identifier'][sample_id]}",
            "method": r.get("method"),
            "method_subtype": r.get("method_subtype"),
            "value_raw": r.get("value_raw"),
            "value": r.get("value"),
            "value_type": r.get("value_type"),
            "unit": r.get("unit"),
            "origin": r.get("origin"),
            "conditions": format_conditions(r.get("conditions")),
            "description": r.get("description"),
            "check": check_tg_record(r),
        })
    if note:
        print(note)

ctx_tg_records = pd.DataFrame(rows)
if not ctx_tg_records.empty:
    dup = ctx_tg_records.groupby(["value_raw", "method", "conditions"]).sample_id.transform("nunique") > 1
    ctx_tg_records.loc[dup, "check"] = (ctx_tg_records.loc[dup, "check"] + "; assigned to several samples").str.strip("; ")
    print(f"\nTg values ({len(ctx_tg_records)}):")
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(ctx_tg_records)
if CTX_RUN_ALL_SAMPLES:
    summary = ctx_samples[["sample_id", "name", "identifier"]].copy()
    values = (ctx_tg_records.groupby("sample_id").value_raw.apply(lambda v: "; ".join(v))
              if not ctx_tg_records.empty else pd.Series(dtype=str))
    summary["tg"] = summary.sample_id.map(values).fillna("")
    print("\nTg per sample:")
    with pd.option_context("display.max_rows", None):
        display(summary)


ID000732356: Ring-opening polymerization of six-membered cyclic hybrid dimers composed of an oxoester and thioester
29 samples in paper; running 1 request(s)

=== 1751759: poly1 | Table 2 Entry 1 ===
--- Reasoning ---
Let me analyze the target sample and the paper to find Tg values.

Target sample: poly1, Table 2 Entry 1
- Linear polymer composed of alternating oxoester-thioester units derived from monomer 1
- Synthesized via anionic ring-opening polymerization in CH2Cl2 at room temperature
- Using 2-ethylhexyl thioglycolate (6 mol%) as initiator and 2,6-lutidine (5 mol%) as catalyst
- Sampled at 1 day (28% conversion, Mn 1050, PDI 1.13) and 3 days (51% conversion, Mn 1900, PDI 1.12)

Now let me look at the thermal properties section and Table 6:

Table 6. Thermal properties of the polymers:
| | Mn | PDI | Tg (°C) | Td10 (°C) |
| poly(rac-thiolactide) | 3860 | 1.5 | −1 ~ 0 | 162 |
| poly(rac-lactide) | – | – | 35.5 ~ 65 | 250 |
| poly1 | 3300 | 1.82 | 3.7 | 194 |
| poly2 | 6100 | 1.15 